In [82]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.metrics import silhouette_score,davies_bouldin_score

## Configuration

In [83]:
RANDOM_STATE=42
ROOT_DIR = Path().cwd().parent
DATASET_DIR = ROOT_DIR / "dataset"
OUT_DIR= DATASET_DIR / "processed" 
CSV_FILE_PATH = OUT_DIR / "clean_athlete_events.csv"
OUT_CSV_FILE_PATH = OUT_DIR / "classification_athlete_events.csv"
print(ROOT_DIR,"\n",DATASET_DIR,"\n",CSV_FILE_PATH,"\n",OUT_CSV_FILE_PATH)

/home/mdev/Documents/ml/olympic_medals_prediction 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset/processed/clean_athlete_events.csv 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset/processed/classification_athlete_events.csv


## Load dataset

In [84]:
df = pd.read_csv(CSV_FILE_PATH)

In [85]:
df = df.drop(columns=['Unnamed: 0'])
df.head()

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
0,CHN,1,1932,1,0.000000,22.0,175.0,70.0,0.0,0.0,1,2,0
1,CHN,1,1936,54,0.037037,24.0,175.0,69.0,0.0,0.0,7,27,0
2,CHN,1,1948,31,0.032258,26.0,175.0,70.0,0.0,0.0,6,12,0
3,CHN,1,1952,1,0.000000,23.0,175.0,70.0,0.0,0.0,1,1,0
4,CHN,1,1984,215,0.386047,23.0,173.0,66.0,0.0,0.0,19,105,32


In [86]:
df.isna().sum()

noc                           0
season                        0
year                          0
athletes                      0
athletes_female_percentage    0
avg_age                       0
agv_height                    0
avg_weight                    0
prev_medals                   0
prev_gold_medals              0
sports                        0
events                        0
total_medals                  0
dtype: int64

In [87]:
X = df.drop(columns=['noc','athletes_female_percentage'])
X.head()

,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
0,1,1932,1,22.0,175.0,70.0,0.0,0.0,1,2,0
1,1,1936,54,24.0,175.0,69.0,0.0,0.0,7,27,0
2,1,1948,31,26.0,175.0,70.0,0.0,0.0,6,12,0
3,1,1952,1,23.0,175.0,70.0,0.0,0.0,1,1,0
4,1,1984,215,23.0,173.0,66.0,0.0,0.0,19,105,32


## Feature Scaling

In [88]:
scaler = RobustScaler()

In [89]:
exclude_cols = ['season']
to_scale_cols = [ x for x in X.columns.tolist() if x not in exclude_cols ]
X[to_scale_cols] = scaler.fit_transform(X[to_scale_cols])

In [90]:
X

,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
0,1,-1.40,-0.285714,-1.000000,0.00,0.000000,0.0,0.0,-0.500,-0.354839,0.0
1,1,-1.30,0.795918,-0.333333,0.00,-0.166667,0.0,0.0,0.250,0.451613,0.0
2,1,-1.00,0.326531,0.333333,0.00,0.000000,0.0,0.0,0.125,-0.032258,0.0
3,1,-0.90,-0.285714,-0.666667,0.00,0.000000,0.0,0.0,-0.500,-0.387097,0.0
4,1,-0.10,4.081633,-0.666667,-0.50,-0.666667,0.0,0.0,1.750,2.967742,8.0
...,...,...,...,...,...,...,...,...,...,...,...
3832,0,0.65,-0.265306,-2.000000,1.25,0.500000,0.0,0.0,-0.500,-0.322581,0.0
3833,0,0.45,-0.285714,-2.000000,1.25,0.666667,0.0,0.0,-0.500,-0.322581,0.0
3834,0,0.55,-0.285714,-0.666667,1.25,0.666667,0.0,0.0,-0.500,-0.354839,0.0
3835,0,0.65,-0.285714,-1.666667,-3.00,-2.333333,0.0,0.0,-0.500,-0.354839,0.0


## Elbow method picking number of clusters

In [91]:
for i in range(1,11):
    km = KMeans(n_clusters=i,n_init="auto",random_state=RANDOM_STATE)
    km.fit(X)
    print("K:",i,f"SSE: {km.inertia_:.2f}")

K: 1 SSE: 239009.49
K: 2 SSE: 113680.19
K: 3 SSE: 67199.69
K: 4 SSE: 57635.64
K: 5 SSE: 53552.34
K: 6 SSE: 48149.72
K: 7 SSE: 44562.22
K: 8 SSE: 37636.94
K: 9 SSE: 34433.75
K: 10 SSE: 32421.93


## Model training and predicting

In [92]:
K=3
k_means = KMeans(n_clusters=K,n_init="auto",random_state=RANDOM_STATE)

In [93]:
km_labels = k_means.fit_predict(X)
km_labels

array([0, 0, 0, ..., 0, 0, 0], shape=(3837,), dtype=int32)

In [94]:
ag = AgglomerativeClustering(n_clusters=K,linkage="ward")
ag_labels = ag.fit_predict(X)

In [95]:
ag_labels

array([0, 0, 0, ..., 0, 0, 0], shape=(3837,))

## Display model metrics

In [96]:
def display_metrics(name:str,silhouette:float,davies_bouldin:float):
    print("Metrics of:",name)
    print(f"Silhouette Score : {silhouette:.2f} (closer to +1 is better)")
    print(f"Davies-Bouldin   : {davies_bouldin:.2f} (lower is better)")

In [97]:
display_metrics("Kmeans",float(silhouette_score(X,km_labels)),davies_bouldin_score(X,km_labels))

Metrics of: Kmeans
Silhouette Score : 0.69 (closer to +1 is better)
Davies-Bouldin   : 0.67 (lower is better)


In [98]:
display_metrics("AgglomerativeClustering",float(silhouette_score(X,ag_labels)),davies_bouldin_score(X,ag_labels))

Metrics of: AgglomerativeClustering
Silhouette Score : 0.70 (closer to +1 is better)
Davies-Bouldin   : 0.63 (lower is better)


## Kmeans centroids view

In [99]:
km_centroids = k_means.cluster_centers_
km_centroids

array([[ 7.19466202e-01, -1.37655933e-01,  2.68408495e-01,
         1.41959192e-01, -9.09486510e-02,  3.29755343e-02,
         4.09824969e-01,  3.29852045e-01,  7.86553525e-02,
         2.14707506e-01,  4.26602843e-01],
       [ 8.25072886e-01, -2.70845481e-01,  3.99565657e+00,
         4.89795918e-01,  1.88775510e-01,  1.80272109e-01,
         7.39067055e+00,  7.14577259e+00,  1.67602041e+00,
         3.18894009e+00,  6.42055394e+00],
       [ 1.00000000e+00, -3.79787234e-01,  7.55840208e+00,
         3.33333333e-01,  4.84042553e-01,  3.15602837e-01,
         3.05177305e+01,  3.62127660e+01,  2.21542553e+00,
         5.26698696e+00,  2.24148936e+01]])

In [100]:
unscaled_centroids = scaler.inverse_transform(km_centroids[:,1:])
season = km_centroids[:,0:1]
km_original_centroids =  np.hstack([season,unscaled_centroids])
km_original_centroids

array([[7.19466202e-01, 1.98249376e+03, 2.81520162e+01, 2.54258776e+01,
        1.74636205e+02, 7.01978532e+01, 1.22947491e+00, 3.29852045e-01,
        5.62924282e+00, 1.96559327e+01, 1.70641137e+00],
       [8.25072886e-01, 1.97716618e+03, 2.10787172e+02, 2.64693878e+01,
        1.75755102e+02, 7.10816327e+01, 2.21720117e+01, 7.14577259e+00,
        1.84081633e+01, 1.11857143e+02, 2.56822157e+01],
       [1.00000000e+00, 1.97280851e+03, 3.85361702e+02, 2.60000000e+01,
        1.76936170e+02, 7.18936170e+01, 9.15531915e+01, 3.62127660e+01,
        2.27234043e+01, 1.76276596e+02, 8.96595745e+01]])

## Clusters 0 low(underdogs) noc 1 middle noc 2 top noc

In [101]:
df_centers = pd.DataFrame(km_original_centroids,columns=X.columns.to_list())
df_centers.index.name = "Cluster"
df_centers

,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
Cluster,,,,,,,,,,,
0,0.719466,1982.493763,28.152016,25.425878,174.636205,70.197853,1.229475,0.329852,5.629243,19.655933,1.706411
1,0.825073,1977.166181,210.787172,26.469388,175.755102,71.081633,22.172012,7.145773,18.408163,111.857143,25.682216
2,1.000000,1972.808511,385.361702,26.000000,176.936170,71.893617,91.553191,36.212766,22.723404,176.276596,89.659574


## Apply cluster labels

In [102]:
df['km_cluster'] = km_labels.astype(int)
df['ag_cluster'] = ag_labels.astype(int)
df.head()

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals,km_cluster,ag_cluster
0,CHN,1,1932,1,0.000000,22.0,175.0,70.0,0.0,0.0,1,2,0,0,0
1,CHN,1,1936,54,0.037037,24.0,175.0,69.0,0.0,0.0,7,27,0,0,0
2,CHN,1,1948,31,0.032258,26.0,175.0,70.0,0.0,0.0,6,12,0,0,0
3,CHN,1,1952,1,0.000000,23.0,175.0,70.0,0.0,0.0,1,1,0,0,0
4,CHN,1,1984,215,0.386047,23.0,173.0,66.0,0.0,0.0,19,105,32,0,0


## Sanity check

In [180]:
sample_idx = np.random.randint(0,df.shape[0],size=3)
sanity = df.loc[sample_idx]
sanity[['prev_medals','total_medals','ag_cluster','km_cluster']]

,prev_medals,total_medals,ag_cluster,km_cluster
93,101.0,110,1,2
436,0.0,0,0,0
3096,0.0,0,0,0


In [181]:
df.to_csv(OUT_CSV_FILE_PATH,index=False)